
Entry: Close > BBU AND RSI < 70

Exit: RSI < 30 OR Close < BBL

# Python Code


In [16]:
!pip install pandas pandas-ta requests plotly -q

import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# ==========================================
# GLOBAL CONFIGURATION
# ==========================================
CONFIG = {
    'SYMBOLS': ['SOLUSDT', 'ADAUSDT'],
    'INTERVAL': '1h',
    'START_DATE': '01/10/2021',
    'END_DATE': '01/10/2022',
    'INITIAL_CAPITAL': 10000.0,
    'POSITION_SIZE_PCT': 0.10,
    'FEE_RATE': 0.0004,
    'SLIPPAGE': 0.0002
}

# ==========================================
# MODULE 1: DATA HANDLER
# ==========================================
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        req_symbol = self.symbol

        print(f"Checking connection for {req_symbol}...")
        try:
            test = requests.get(f"{base_url}/api/v3/time", timeout=5)
            if test.status_code == 451:
                print("US IP detected. Switching to Binance.US...")
                base_url = "https://api.binance.us"
                if req_symbol == "ETHUSDT": req_symbol = "ETHUSD"
                if req_symbol == "BNBUSDT": req_symbol = "BNBUSD"
        except Exception:
            base_url = "https://api.binance.us"
            if req_symbol == "ETHUSDT": req_symbol = "ETHUSD"
            if req_symbol == "BNBUSDT": req_symbol = "BNBUSD"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)

        if self.end_date == datetime.now().strftime("%d/%m/%Y"):
            end_time = int(time.time() * 1000)
        else:
            end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
            end_time = int(end_dt.timestamp() * 1000)

        all_klines =[]
        print(f"Downloading {req_symbol} ({self.interval})...")

        while start_time < end_time:
            params = {"symbol": req_symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except Exception: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df.astype(float)
        print(f"Total: {len(df)} candles fetched.")
        return df

# ==========================================
# MODULE 2: STRATEGY LOGIC
# ==========================================
class ShortBollingerRsiStrategy:
    def __init__(self):
        # Tham số cho Bollinger Bands và RSI
        self.bb_length = 20
        self.bb_std = 2.0
        self.rsi_length = 14
        self.rsi_threshold = 70

    def calculate_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        # Tính RSI
        df['RSI'] = ta.rsi(df['Close'], length=self.rsi_length)

        # Tính Bollinger Bands
        bb = ta.bbands(df['Close'], length=self.bb_length, std=self.bb_std)
        # Thư viện pandas_ta trả về DataFrame có nhiều cột, ta lấy cột Lower và Upper
        df['BBL'] = bb.iloc[:, 0] # Dải dưới
        df['BBU'] = bb.iloc[:, 2] # Dải trên

        df.dropna(inplace=True)
        return df

# ==========================================
# MODULE 3: BACKTEST ENGINE (SHORT ONLY)
# ==========================================
class BacktestEngine:
    def __init__(self, df, strategy, config):
        self.df = strategy.calculate_indicators(df.copy())
        self.strategy = strategy
        self.config = config

        self.capital = config['INITIAL_CAPITAL']
        self.invested_amount = 0.0
        self.position = 0
        self.entry_price = 0.0
        self.trades = []

        self.equity_curve = np.zeros(len(self.df))

    def run(self):
        O = self.df['Open'].values
        C = self.df['Close'].values
        times = self.df.index

        # Caching values
        rsi = self.df['RSI'].values
        bbl = self.df['BBL'].values
        bbu = self.df['BBU'].values

        fee = self.config['FEE_RATE']
        slip = self.config['SLIPPAGE']
        pos_size_pct = self.config['POSITION_SIZE_PCT']

        s = self.strategy

        self.equity_curve[0] = self.capital

        for i in range(1, len(self.df)):

            # 1. EXIT LOGIC (SHORT ONLY)
            if self.position == -1:
                exit_signal = False
                reason = ""

                # Exit Condition: RSI < 70 AND Close > BB_Lower
                if rsi[i-1] < 30 and C[i-1] < bbl[i-1]:
                    exit_signal = True
                    reason = "TP/Exit Condition Met"


                if exit_signal:
                    # Khi đóng lệnh Short, ta phải "Mua lại" (Buy to cover)
                    # Nên trượt giá sẽ làm giá thoát CAO HƠN giá Open
                    real_exit = O[i] * (1 + slip)

                    # Công thức PnL cho lệnh Short (Giá vào - Giá ra)
                    pnl_pct = (self.entry_price - real_exit) / self.entry_price
                    net_pnl_pct = pnl_pct - (fee * 2)
                    pnl_amt = self.invested_amount * net_pnl_pct
                    self.capital += pnl_amt

                    self.trades.append({
                        'Date': times[i], 'Type': 'Short',
                        'Reason': reason,
                        'Entry': self.entry_price, 'Exit': real_exit,
                        'PnL_Amt': pnl_amt, 'Equity': self.capital
                    })

                    self.position = 0
                    self.invested_amount = 0.0

            # 2. ENTRY LOGIC (SHORT ONLY)
            if self.position == 0:
                # Entry Condition: Close > BB_Upper AND RSI < 70
                if C[i-1] > bbu[i-1] and rsi[i-1] < s.rsi_threshold:
                    self.position = -1
                    # Khi mở lệnh Short, ta "Bán khống"
                    # Nên trượt giá sẽ làm giá khớp THẤP HƠN giá Open
                    self.entry_price = O[i] * (1 - slip)
                    self.invested_amount = self.capital * pos_size_pct

            # 3. TÍNH LÃI/LỖ TẠM TÍNH (UNREALIZED PNL CHO SHORT)
            current_equity = self.capital
            if self.position == -1:
                # Với lệnh Short: Giá giảm -> Lãi ; Giá tăng -> Lỗ
                unrealized_pnl_pct = (self.entry_price - C[i]) / self.entry_price
                unrealized_pnl = self.invested_amount * unrealized_pnl_pct
                current_equity += unrealized_pnl

            self.equity_curve[i] = current_equity

        return pd.Series(self.equity_curve, index=self.df.index), pd.DataFrame(self.trades)

# ==========================================
# MODULE 4: PERFORMANCE EVALUATOR
# ==========================================
class Evaluator:
    @staticmethod
    def generate_report(equity, trades_df, initial_capital, symbol, interval, pos_size_pct):
        if trades_df.empty:
            print(f"No trades executed for {symbol}!")
            return

        final_equity = equity.iloc[-1]
        net_pnl = final_equity - initial_capital
        total_return = net_pnl / initial_capital

        rolling_max = equity.cummax()
        drawdown = (equity - rolling_max) / rolling_max
        max_dd = drawdown.min()

        wins = trades_df[trades_df['PnL_Amt'] > 0]
        losses = trades_df[trades_df['PnL_Amt'] <= 0]
        winrate = len(wins) / len(trades_df) if len(trades_df) > 0 else 0
        profit_factor = wins['PnL_Amt'].sum() / abs(losses['PnL_Amt'].sum()) if len(losses) > 0 and losses['PnL_Amt'].sum() != 0 else 0

        days = (equity.index[-1] - equity.index[0]).days
        cagr = ((final_equity / initial_capital) ** (365.0 / days)) - 1 if days > 0 else 0

        # Hàm nhận diện số nến trong 1 năm
        def get_annual_factor(tf):
            if tf == '15m': return 365 * 24 * 4
            elif tf == '1h': return 365 * 24
            elif tf == '4h': return 365 * 6
            elif tf == '1d': return 365
            return 365 * 24 # Default 1h

        annual_factor = get_annual_factor(interval)
        returns = equity.pct_change().dropna()
        sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(annual_factor) if returns.std() != 0 else 0


        print("")
        print(f"========== PERFORMANCE REPORT: {symbol} ({interval}) ==========")
        print(f"{'Strategy:':>20} Shorting BB Upper + RSI")
        print(f"{'Position Sizing:':>20} {pos_size_pct*100}% of Capital")
        print(f"{'Initial Capital:':>20} ${initial_capital:,.2f}")
        print(f"{'Final Equity:':>20} ${final_equity:,.2f}")
        print(f"{'Net PnL:':>20} ${net_pnl:,.2f}")
        print(f"{'Total Return:':>20} {total_return:.2%}")
        print(f"{'CAGR (Annual):':>20} {cagr:.2%}")
        print(f"{'Max Drawdown:':>20} {max_dd:.2%}")
        print(f"{'Sharpe Ratio:':>20} {sharpe_ratio:.2f}")
        print(f"{'Total Trades:':>20} {len(trades_df)}")
        print(f"{'Winrate:':>20} {winrate:.2%}")
        print(f"{'Profit Factor:':>20} {profit_factor:.2f}")
        print("====================================================\n")

        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.08,
            row_heights=[0.7, 0.3]
        )

        fig.add_trace(go.Scatter(
            x=equity.index, y=equity.values,
            mode="lines", name="Equity",
            line=dict(color="#2962ff", width=1.5)
        ), row=1, col=1)

        fig.add_trace(go.Scatter(
            x=drawdown.index, y=drawdown.values,
            mode="lines", name="Drawdown",
            line=dict(color="#d32f2f", width=1),
            fill='tozeroy', fillcolor='rgba(211, 47, 47, 0.2)'
        ), row=2, col=1)

        fig.update_layout(
            title=f"Portfolio Performance: {symbol} ({interval})",
            template="plotly_white",
            height=650,
            margin=dict(l=40, r=40, t=60, b=40),
            showlegend=False
        )

        fig.update_xaxes(showgrid=True, gridcolor='#e6e6e6', zeroline=False)
        fig.update_yaxes(title_text="Balance ($)", tickprefix="$", showgrid=True, gridcolor='#e6e6e6', zeroline=False, row=1, col=1)
        fig.update_yaxes(title_text="Drawdown %", tickformat=".1%", showgrid=True, gridcolor='#e6e6e6', zeroline=True, zerolinecolor='#cccccc', row=2, col=1)

        fig.show()

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    for symbol in CONFIG['SYMBOLS']:
        print(f"\n---> Starting Backtest for: {symbol} <---")
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()

        if not df_raw.empty:
            # Sử dụng chiến lược mới
            my_strategy = ShortBollingerRsiStrategy()

            engine = BacktestEngine(df_raw, my_strategy, CONFIG)
            equity, trades = engine.run()

            Evaluator.generate_report(
                equity,
                trades,
                CONFIG['INITIAL_CAPITAL'],
                symbol,
                CONFIG['INTERVAL'],
                CONFIG['POSITION_SIZE_PCT']
            )


---> Starting Backtest for: SOLUSDT <---
Checking connection for SOLUSDT...
Total: 8784 candles fetched.

========== PERFORMANCE REPORT: SOLUSDT (1h) ==========
           Strategy: Shorting BB Upper + RSI
    Position Sizing: 10.0% of Capital
    Initial Capital: $10,000.00
       Final Equity: $11,041.22
            Net PnL: $1,041.22
       Total Return: 10.41%
      CAGR (Annual): 10.41%
       Max Drawdown: -7.34%
       Sharpe Ratio: 1.04
       Total Trades: 50
            Winrate: 70.00%
      Profit Factor: 1.50




---> Starting Backtest for: ADAUSDT <---
Checking connection for ADAUSDT...
Total: 8784 candles fetched.

========== PERFORMANCE REPORT: ADAUSDT (1h) ==========
           Strategy: Shorting BB Upper + RSI
    Position Sizing: 10.0% of Capital
    Initial Capital: $10,000.00
       Final Equity: $11,266.31
            Net PnL: $1,266.31
       Total Return: 12.66%
      CAGR (Annual): 12.66%
       Max Drawdown: -5.45%
       Sharpe Ratio: 1.50
       Total Trades: 49
            Winrate: 75.51%
      Profit Factor: 2.14



# Go Code

In [ ]:
package main

import (
	"encoding/json"
	"fmt"
	"math"
	"net/http"
	"strconv"
	"strings"
	"time"
)

// ==========================================
// GLOBAL CONFIGURATION
// ==========================================
type Config struct {
	Symbols         []string
	Interval        string
	StartDate       string
	EndDate         string
	InitialCapital  float64
	PositionSizePct float64
	FeeRate         float64
	Slippage        float64
}

var globalConfig = Config{
	Symbols:         []string{"SOLUSDT", "ADAUSDT"},
	Interval:        "1h",
	StartDate:       "01/10/2021",
	EndDate:         "01/10/2022",
	InitialCapital:  10000.0,
	PositionSizePct: 0.10,
	FeeRate:         0.0004,
	Slippage:        0.0002,
}

// ==========================================
// CORE DATA STRUCTURES
// ==========================================
type Candle struct {
	Timestamp time.Time
	Open      float64
	High      float64
	Low       float64
	Close     float64
	Volume    float64
}

type IndicatorData struct {
	Candle
	RSI float64
	BBL float64
	BBU float64
}

type Trade struct {
	Date   time.Time
	Type   string
	Reason string
	Entry  float64
	Exit   float64
	PnL    float64
	Equity float64
}

// ==========================================
// MODULE 1: DATA HANDLER
// ==========================================
func fetchBinanceData(symbol, interval, startDate, endDate string) []Candle {
	layout := "02/01/2006"
	startDT, _ := time.Parse(layout, startDate)
	endDT, _ := time.Parse(layout+" 15:04:05", endDate+" 23:59:59")

	startTime := startDT.UnixMilli()
	endTime := endDT.UnixMilli()

	baseURL := "https://api.binance.com/api/v3/klines"
	var allCandles []Candle

	fmt.Printf("Downloading %s (%s)...\n", symbol, interval)

	for startTime < endTime {
		url := fmt.Sprintf("%s?symbol=%s&interval=%s&startTime=%d&endTime=%d&limit=1000",
			baseURL, symbol, interval, startTime, endTime)

		resp, err := http.Get(url)
		if err != nil || resp.StatusCode != 200 {
			break
		}

		var rawData [][]interface{}
		json.NewDecoder(resp.Body).Decode(&rawData)
		resp.Body.Close()

		if len(rawData) == 0 {
			break
		}

		for _, row := range rawData {
			ts := int64(row[0].(float64))
			open, _ := strconv.ParseFloat(row[1].(string), 64)
			high, _ := strconv.ParseFloat(row[2].(string), 64)
			low, _ := strconv.ParseFloat(row[3].(string), 64)
			close, _ := strconv.ParseFloat(row[4].(string), 64)
			vol, _ := strconv.ParseFloat(row[5].(string), 64)

			allCandles = append(allCandles, Candle{
				Timestamp: time.UnixMilli(ts),
				Open:      open,
				High:      high,
				Low:       low,
				Close:     close,
				Volume:    vol,
			})
		}

		lastTS := int64(rawData[len(rawData)-1][0].(float64))
		if startTime == lastTS+1 {
			break
		}
		startTime = lastTS + 1
		time.Sleep(50 * time.Millisecond)
	}

	fmt.Printf("Total: %d candles fetched.\n", len(allCandles))
	return allCandles
}

// ==========================================
// MODULE 2: STRATEGY LOGIC (INDICATORS)
// ==========================================
func calculateIndicators(candles []Candle, rsiPeriod, bbPeriod int, bbStd float64) []IndicatorData {
	n := len(candles)
	data := make([]IndicatorData, n)

	// Calculate RSI (Wilder's Smoothing)
	gains := make([]float64, n)
	losses := make([]float64, n)

	for i := 1; i < n; i++ {
		change := candles[i].Close - candles[i-1].Close
		if change > 0 {
			gains[i] = change
		} else {
			losses[i] = -change
		}
	}

	var avgGain, avgLoss float64
	for i := 1; i <= rsiPeriod && i < n; i++ {
		avgGain += gains[i]
		avgLoss += losses[i]
	}
	avgGain /= float64(rsiPeriod)
	avgLoss /= float64(rsiPeriod)

	for i := 0; i < n; i++ {
		data[i].Candle = candles[i]

		if i == rsiPeriod {
			rs := avgGain / avgLoss
			if avgLoss == 0 {
				data[i].RSI = 100
			} else {
				data[i].RSI = 100 - (100 / (1 + rs))
			}
		} else if i > rsiPeriod {
			avgGain = (avgGain*float64(rsiPeriod-1) + gains[i]) / float64(rsiPeriod)
			avgLoss = (avgLoss*float64(rsiPeriod-1) + losses[i]) / float64(rsiPeriod)
			rs := avgGain / avgLoss
			if avgLoss == 0 {
				data[i].RSI = 100
			} else {
				data[i].RSI = 100 - (100 / (1 + rs))
			}
		}

		// Calculate Bollinger Bands
		if i >= bbPeriod-1 {
			sum := 0.0
			for j := i - bbPeriod + 1; j <= i; j++ {
				sum += candles[j].Close
			}
			sma := sum / float64(bbPeriod)

			varianceSum := 0.0
			for j := i - bbPeriod + 1; j <= i; j++ {
				varianceSum += math.Pow(candles[j].Close-sma, 2)
			}
			stdDev := math.Sqrt(varianceSum / float64(bbPeriod))

			data[i].BBU = sma + (stdDev * bbStd)
			data[i].BBL = sma - (stdDev * bbStd)
		}
	}
	return data
}

// ==========================================
// MODULE 3: BACKTEST ENGINE (SHORT ONLY)
// ==========================================
func runBacktest(data []IndicatorData, config Config) ([]float64, []Trade) {
	capital := config.InitialCapital
	investedAmount := 0.0
	position := 0 // 0: Flat, -1: Short
	entryPrice := 0.0

	var trades []Trade
	equityCurve := make([]float64, len(data))
	equityCurve[0] = capital

	for i := 1; i < len(data); i++ {
		prev := data[i-1]
		curr := data[i]

		// 1. EXIT LOGIC (SHORT ONLY)
		if position == -1 {
			exitSignal := false
			reason := ""

			// Exit Condition: RSI < 30 AND Close < BB_Lower
			if prev.RSI < 30 && prev.Close < prev.BBL {
				exitSignal = true
				reason = "TP/Exit Condition Met"
			}

			if exitSignal {
				realExit := curr.Open * (1 + config.Slippage)
				pnlPct := (entryPrice - realExit) / entryPrice
				netPnlPct := pnlPct - (config.FeeRate * 2)
				pnlAmt := investedAmount * netPnlPct
				capital += pnlAmt

				trades = append(trades, Trade{
					Date:   curr.Timestamp,
					Type:   "Short",
					Reason: reason,
					Entry:  entryPrice,
					Exit:   realExit,
					PnL:    pnlAmt,
					Equity: capital,
				})

				position = 0
				investedAmount = 0.0
			}
		}

		// 2. ENTRY LOGIC (SHORT ONLY)
		if position == 0 {
			// Entry Condition: Close > BB_Upper AND RSI < 70
			// NOTE: Replicated exactly as python version logic
			if prev.Close > prev.BBU && prev.RSI < 70 && prev.BBU > 0 {
				position = -1
				entryPrice = curr.Open * (1 - config.Slippage)
				investedAmount = capital * config.PositionSizePct
			}
		}

		// 3. UNREALIZED PNL
		currentEquity := capital
		if position == -1 {
			unrealizedPnlPct := (entryPrice - curr.Close) / entryPrice
			currentEquity += investedAmount * unrealizedPnlPct
		}
		equityCurve[i] = currentEquity
	}

	return equityCurve, trades
}

// ==========================================
// MODULE 4: EVALUATOR
// ==========================================
func evaluatePerformance(symbol string, equity []float64, trades []Trade, config Config) {
	if len(trades) == 0 {
		fmt.Printf("\nNo trades executed for %s!\n", symbol)
		return
	}

	finalEquity := equity[len(equity)-1]
	netPnl := finalEquity - config.InitialCapital
	totalReturn := netPnl / config.InitialCapital

	maxEquity := equity[0]
	maxDD := 0.0

	for _, eq := range equity {
		if eq > maxEquity {
			maxEquity = eq
		}
		dd := (eq - maxEquity) / maxEquity
		if dd < maxDD {
			maxDD = dd
		}
	}

	wins, losses := 0, 0
	grossProfit, grossLoss := 0.0, 0.0

	for _, t := range trades {
		if t.PnL > 0 {
			wins++
			grossProfit += t.PnL
		} else {
			losses++
			grossLoss += math.Abs(t.PnL)
		}
	}

	winrate := float64(wins) / float64(len(trades))
	profitFactor := 0.0
	if grossLoss > 0 {
		profitFactor = grossProfit / grossLoss
	} else {
		profitFactor = math.Inf(1)
	}

	fmt.Printf("\n========== PERFORMANCE REPORT: %s ==========\n", symbol)
	fmt.Printf("%-20s Shorting BB Upper + RSI\n", "Strategy:")
	fmt.Printf("%-20s %.0f%% of Capital\n", "Position Sizing:", config.PositionSizePct*100)
	fmt.Printf("%-20s $%.2f\n", "Initial Capital:", config.InitialCapital)
	fmt.Printf("%-20s $%.2f\n", "Final Equity:", finalEquity)
	fmt.Printf("%-20s $%.2f\n", "Net PnL:", netPnl)
	fmt.Printf("%-20s %.2f%%\n", "Total Return:", totalReturn*100)
	fmt.Printf("%-20s %.2f%%\n", "Max Drawdown:", maxDD*100)
	fmt.Printf("%-20s %d\n", "Total Trades:", len(trades))
	fmt.Printf("%-20s %.2f%%\n", "Winrate:", winrate*100)
	fmt.Printf("%-20s %.2f\n", "Profit Factor:", profitFactor)
	fmt.Println(strings.Repeat("=", 50))
}

// ==========================================
// MAIN EXECUTION
// ==========================================
func main() {
	for _, symbol := range globalConfig.Symbols {
		fmt.Printf("\n---> Starting Backtest for: %s <---\n", symbol)

		// 1. Lấy dữ liệu
		candles := fetchBinanceData(symbol, globalConfig.Interval, globalConfig.StartDate, globalConfig.EndDate)

		if len(candles) > 0 {
			// 2. Tính toán Indicator (BB 20 2, RSI 14)
			indicatorData := calculateIndicators(candles, 14, 20, 2.0)

			// 3. Chạy Engine
			equityCurve, trades := runBacktest(indicatorData, globalConfig)

			// 4. In báo cáo
			evaluatePerformance(symbol, equityCurve, trades, globalConfig)
		}
	}
}